# ScavenPred — current grouped nested forward-wrapper reproduction

**Primary, current modelling workflow (10 November 2026).** This notebook executes the versioned repository scripts rather than reproducing the previous frozen model with a hard-coded descriptor panel. It runs grouped nested 5×3 cross-validation and a separately similarity-screened external evaluation. Both use Welch screening, Pearson filtering, VIF and forward-wrapper selection. Sequence groups are isolated in outer folds; reported out-of-fold metrics use predictions on held-out groups. Pre-screening is performed on each **outer training** set before inner wrapper search; inner MCC is therefore used for selection only.

Run from a clone of `agboken/ScavenPred-ACE-iPred` with dependencies installed from `reproducibility/requirements.txt`. The complete experiments can take substantial CPU time. Output files are generated locally, not embedded or fabricated here.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
import pandas as pd
from IPython.display import display

# Resolve repository root when launched from repo root, notebooks/, or another subfolder.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "reproducibility/scripts/grouped_nested_forward_wrapper_cv.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the cloned ScavenPred-ACE-iPred repository.")
REPRO = ROOT / "reproducibility"
DATA = REPRO / "data" / "csv"
OUT = ROOT / "reproduced_grouped_wrapper" / "scavenpred"
OUT.mkdir(parents=True, exist_ok=True)
MODEL = "scavenpred"
print("Repository:", ROOT)
print("Model:", MODEL)

## 1. Restore both training and test datasets

The data archive is reconstructed with a SHA-256 check. The original root Excel files are obsolete and are **not** required.

In [ ]:
subprocess.run([sys.executable, str(REPRO / "scripts/unpack_data.py")], cwd=ROOT, check=True)
print("CSV files:", sorted(p.name for p in DATA.glob("*.csv")))

## 2. Sequence-grouped nested cross-validation

Five outer groups and three inner grouped folds. Welch/Pearson/VIF screening and RF forward-wrapper model selection occur using only outer training data. The selected features may change across outer folds.

In [ ]:
env = os.environ.copy()
env.update(OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
subprocess.run([
    sys.executable, str(REPRO / "scripts/grouped_nested_forward_wrapper_cv.py"),
    "--data", str(DATA), "--out", str(OUT),
    "--model", MODEL, "--inner-trees", "40", "--candidate-limit", "12"
], cwd=ROOT, env=env, check=True)

In [ ]:
cv = json.loads((OUT / f"{MODEL}_wrapper_group_report.json").read_text())
display(pd.DataFrame([{
    "Peptides": cv.get("n", cv.get("n_samples", "")),
    "Accuracy": cv["oof_accuracy"], "MCC": cv["oof_mcc"],
    "ROC-AUC": cv["oof_roc_auc"], "PR-AUC": cv["oof_pr_auc"],
    "Confusion matrix": str(cv["confusion_matrix"])
}]))
print("Outer-fold feature selections:")
for i, fold in enumerate(cv.get("folds", []), 1):
    print(f"Fold {i}:", fold.get("selected_features", fold.get("features", fold)))
oof = pd.read_csv(OUT / f"{MODEL}_wrapper_group_oof.csv")
display(oof.head())

## 3. Final training and similarity-screened external evaluation

After training-only feature selection on the entire training set, external sequences similar to training sequences under the declared rules are excluded by **sequence alone**. Neither external activity labels nor external descriptor values enter feature selection. Final descriptor panels are extracted from the resulting report.

In [ ]:
subprocess.run([
    sys.executable, str(REPRO / "scripts/forward_wrapper_screened_external.py"),
    "--data", str(DATA), "--out", str(OUT),
    "--model", MODEL, "--inner-trees", "40", "--candidate-limit", "12"
], cwd=ROOT, env=env, check=True)

In [ ]:
external = json.loads((OUT / f"{MODEL}_wrapper_external_report.json").read_text())
print("Selected final training descriptors:", external["selected_features"])
display(pd.DataFrame([{
    "Training peptides": external["training_n"],
    "External evaluated": external["external_evaluated"],
    "Accuracy": external["accuracy"], "MCC": external["mcc"],
    "ROC-AUC": external["roc_auc"], "PR-AUC": external["pr_auc"],
    "Confusion matrix": str(external["confusion_matrix"])
}]))
display(pd.read_csv(OUT / f"{MODEL}_wrapper_external_predictions.csv").head())

## 4. Independent MCC arithmetic check

Compute MCC directly from prediction records to ensure it is not inadvertently averaged across outer folds.

In [ ]:
from sklearn.metrics import matthews_corrcoef
assert abs(matthews_corrcoef(oof["actual"], oof["prediction"]) - cv["oof_mcc"]) < 1e-10
scored = pd.read_csv(OUT / f"{MODEL}_wrapper_external_predictions.csv").dropna(subset=["prediction"])
assert abs(matthews_corrcoef(scored["actual"].astype(int), scored["prediction"].astype(int)) - external["mcc"]) < 1e-10
print("Both MCC values verified directly from peptide-level predictions.")

## 5. Interpretation and provenance

Current result records: [grouped CV](../verification/2026-11-10_grouped_nested_forward_wrapper.md), [wrapper-consistent external](../verification/2026-11-10_grouped_forward_wrapper_external.md). The historic SelectKBest and frozen-model notebooks are retained under [archive/](archive/) for audit only. The original frozen feature lists and deployed web models are **not** the current final wrapper-selected feature panels. Do not substitute their metrics for these experiments.